# SPR-01 — Exploratory Data Analysis (Final Dataset)

**Ticket:** SPR-01
**Owner:** Prithila
**Depends on:** none
**Folder:** `notebooks/03_income_classification/`
**Output:** distribution report + figures in `results/figures/`, findings summary in `docs/findings/`

**A note on scikit-plot:** scikit-plot only covers model-evaluation plots (confusion matrix, ROC, precision-recall, learning curves), not raw-data exploration. There is no dataset yet to model at this stage, so it has nothing to plot here. It's used correctly downstream in SPR-04. This notebook instead applies uniform IEEE-grade static styling (high DPI, consistent fonts, colorblind-safe palette) across every plot by hand, which achieves the same publication-quality goal.

**Column names below follow the project's known schema** (`age`, `education_years`, `digital_index`, `income_class`, etc.). This specific file (`final_dataset_preprocessed_distributed_social_class.csv`) may have renamed or restructured columns, the filename suggests income/social class handling may differ. **Run the column-check cell in Section 2 first and adjust names before continuing.**

## 1. Download dataset from Hugging Face

In [ ]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/final_dataset_preprocessed_distributed_social_class.csv

## 2. Load and inspect structure
Run this first and actually read the output before anything else in this notebook. Column names, dtypes, and the target column name are confirmed here, not assumed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os

np.random.seed(42)
random.seed(42)

df = pd.read_csv('final_dataset_preprocessed_distributed_social_class.csv')

print('Shape:', df.shape)
print()
print('Columns:')
print(list(df.columns))
print()
print('Dtypes:')
print(df.dtypes)

In [ ]:
df.head()

In [ ]:
df.describe(include='all').T

## 3. Publication-quality plot styling (IEEE Access grade)
Applied globally so every figure in this notebook is consistent: high DPI for print, readable font sizes, colorblind-safe palette. Set once here, used everywhere below.

In [ ]:
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.figsize': (8, 5)
})

# Colorblind-safe qualitative palette (Okabe-Ito), works in grayscale print too
IEEE_PALETTE = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9', '#F0E442']
sns.set_palette(IEEE_PALETTE)

os.makedirs('results/figures', exist_ok=True)
os.makedirs('docs/findings', exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(f'results/figures/{name}.png', dpi=300, bbox_inches='tight')
    plt.show()

## 4. Missing values check

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_table = missing_table[missing_table['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing_table if len(missing_table) > 0 else 'No missing values found.')

## 5. Target variable distribution: income_class
This is the single most important chart in this notebook. Confirms the real class imbalance (No_income / Lower / Middle / Upper) that SPR-02 has to handle.

In [ ]:
TARGET_COL = 'income_class'  # TODO: confirm this matches the actual column name from Section 2

class_counts = df[TARGET_COL].value_counts()
class_pct = df[TARGET_COL].value_counts(normalize=True).mul(100).round(2)

print(class_counts)
print()
print(class_pct)

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x=TARGET_COL, order=class_counts.index, color=IEEE_PALETTE[0])
for i, count in enumerate(class_counts.values):
    pct = class_pct.iloc[i]
    ax.text(i, count, f'{count:,}\n({pct}%)', ha='center', va='bottom', fontsize=10)
plt.title('Income Class Distribution')
plt.xlabel('Income Class')
plt.ylabel('Count')
save_fig('target_distribution_income_class')

## 6. Upper-tail distribution check
Requested explicitly for this ticket. The Upper income class is known to be small (around 1% in the project's earlier profiling); this section looks specifically at what that tail looks like across continuous features, since it's the class every model will struggle with most.

In [ ]:
upper_mask = df[TARGET_COL] == 'Upper'  # TODO: confirm exact label spelling from Section 2/5 output
upper_df = df[upper_mask]
non_upper_df = df[~upper_mask]

print(f'Upper class rows: {len(upper_df):,} ({len(upper_df)/len(df)*100:.2f}% of dataset)')
print()
print('Upper class summary statistics (numeric columns):')
upper_df.describe().T

In [ ]:
# Compare key continuous features between Upper and everyone else
continuous_candidates = ['age', 'education_years', 'weekly_workhours', 'digital_index',
                          'social_participation_index']
continuous_features = [c for c in continuous_candidates if c in df.columns]

fig, axes = plt.subplots(1, len(continuous_features), figsize=(5 * len(continuous_features), 5))
if len(continuous_features) == 1:
    axes = [axes]

for ax, col in zip(axes, continuous_features):
    plot_df = df.copy()
    plot_df['is_upper'] = np.where(upper_mask, 'Upper', 'Other')
    sns.boxplot(data=plot_df, x='is_upper', y=col, ax=ax, palette=IEEE_PALETTE)
    ax.set_title(col)
    ax.set_xlabel('')

plt.suptitle('Upper Income Class vs. Rest: Feature Comparison', y=1.02)
save_fig('upper_tail_feature_comparison')

## 7. Continuous feature distributions
One histogram per numeric feature, with a KDE overlay. Consistent styling from Section 3.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Drop obvious one-hot / binary flag columns from this view, they're better shown as bar counts
binary_like = [c for c in numeric_cols if df[c].dropna().isin([0, 1]).all()]
continuous_only = [c for c in numeric_cols if c not in binary_like]

print('Continuous columns:', continuous_only)
print('Binary/flag columns (shown separately below):', binary_like)

In [ ]:
n_cols = 3
n_rows = -(-len(continuous_only) // n_cols)  # ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, continuous_only):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color=IEEE_PALETTE[0])
    ax.set_title(col)

for ax in axes[len(continuous_only):]:
    ax.axis('off')

plt.suptitle('Continuous Feature Distributions', y=1.02)
save_fig('continuous_feature_distributions')

## 8. Binary / flag feature prevalence
Shows what share of the dataset has each binary flag set to 1 (gender_male, disabled, agri_worker, region dummies, etc). More informative than a histogram for 0/1 columns.

In [ ]:
if binary_like:
    prevalence = df[binary_like].mean().sort_values(ascending=False) * 100
    plt.figure(figsize=(8, max(4, len(binary_like) * 0.35)))
    sns.barplot(x=prevalence.values, y=prevalence.index, color=IEEE_PALETTE[1])
    plt.xlabel('% of rows with flag = 1')
    plt.title('Binary Feature Prevalence')
    save_fig('binary_feature_prevalence')
else:
    print('No binary-like columns detected.')

## 9. digital_index by income_class
The core research relationship. This is the descriptive precursor to the MNLogit result already established for this project; it should visually show the same monotonic upward pattern.

In [ ]:
plt.figure(figsize=(7, 5))
order = class_counts.index.tolist()
sns.boxplot(data=df, x=TARGET_COL, y='digital_index', order=order, palette=IEEE_PALETTE)
plt.title('Digital Index by Income Class')
plt.xlabel('Income Class')
plt.ylabel('Digital Index')
save_fig('digital_index_by_income_class')

In [ ]:
# Same relationship as a mean-with-error-bar chart, often clearer for a paper figure than a boxplot
plt.figure(figsize=(7, 5))
sns.pointplot(data=df, x=TARGET_COL, y='digital_index', order=order,
              color=IEEE_PALETTE[0], errorbar='ci', capsize=0.15)
plt.title('Mean Digital Index by Income Class (95% CI)')
plt.xlabel('Income Class')
plt.ylabel('Mean Digital Index')
save_fig('digital_index_by_income_class_pointplot')

## 10. Correlation heatmap (continuous features)
Flags potential redundancy before modeling, e.g. `digital_index` vs `digital_status`, which the project notes flagged as related-but-distinct and worth checking.

In [ ]:
corr = df[continuous_only].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={'label': 'Pearson r'})
plt.title('Correlation Matrix: Continuous Features')
save_fig('correlation_heatmap')

## 11. Region distribution

In [ ]:
region_cols = [c for c in df.columns if c.startswith('region_')]
if region_cols:
    region_counts = df[region_cols].sum().sort_values(ascending=False)
    region_counts.index = region_counts.index.str.replace('region_', '')

    plt.figure(figsize=(8, 5))
    sns.barplot(x=region_counts.values, y=region_counts.index, color=IEEE_PALETTE[2])
    plt.xlabel('Count')
    plt.title('Region Distribution')
    save_fig('region_distribution')
else:
    print('No region_* columns found, check actual column names from Section 2.')

## 12. Duplicate rows check

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes:,} ({n_dupes/len(df)*100:.3f}% of dataset)')

## 13. Summary table for the profiling report
Saved to `docs/findings/`, this is the deliverable SPR-01 actually promises: a written distribution and upper-tail summary other tickets can reference without re-running this notebook.

In [ ]:
summary_lines = []
summary_lines.append(f'# EDA Summary: SIGNAL Final Dataset\n')
summary_lines.append(f'Rows: {df.shape[0]:,}, Columns: {df.shape[1]}\n')
summary_lines.append(f'## Target Class Distribution ({TARGET_COL})\n')
for cls, count in class_counts.items():
    pct = class_pct[cls]
    summary_lines.append(f'- {cls}: {count:,} ({pct}%)\n')
summary_lines.append(f'\n## Upper-Tail Notes\n')
summary_lines.append(f'- Upper class share: {len(upper_df)/len(df)*100:.2f}%\n')
summary_lines.append(f'- Duplicate rows: {n_dupes:,}\n')
summary_lines.append(f'\n_Fill in qualitative observations after reviewing the figures above._\n')

with open('docs/findings/eda_summary.md', 'w') as f:
    f.writelines(summary_lines)

print('Saved docs/findings/eda_summary.md')
print(''.join(summary_lines))

## 14. Written findings
_Fill in after reviewing every figure above. This feeds SPR-08a (Introduction) and SPR-02 (imbalance strategy justification)._

- Most imbalanced class and its share: 
- Any surprising correlations: 
- Any data quality issues found (missingness, duplicates): 
- Confirmed: does digital_index rise monotonically across income class visually, matching the MNLogit result: 